In [ ]:
import sys

sys.path.append("..")
sys.path.insert(0, "../../wildlife-datasets")
import os

import numpy as np
from tqdm import tqdm
from wildlife_datasets.datasets import TurtlewatchEgypt_Master

from animal_re_id.config import load_paths, resolve_path
from animal_re_id.datasets import combine_datasets_twe

In [ ]:
front_flippers = True
min_size = 200
min_score = 0.85

paths = load_paths("TurtlewatchEgypt")
file_name = resolve_path(paths["file_name"])
root_master = resolve_path(paths["root_master"])
root_new = resolve_path(paths["root_new"])
roots_new = paths["roots_new"]

dataset_new = combine_datasets_twe(root_new, roots_new, file_name, img_load="bbox")
dataset_master = TurtlewatchEgypt_Master(root_master, file_name=file_name)

if front_flippers:
    root_results = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_new"
    names = ["flipper_fl", "flipper_fr"]
    new_identities = np.setdiff1d(dataset_new.identities, dataset_master.identities)
else:
    root_results = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_rear"
    names = ["flipper_rl", "flipper_rr"]
    new_identities = dataset_new.identities

os.makedirs(root_results, exist_ok=True)

In [ ]:
dataset_new.df["size_x"] = dataset_new.df["bbox"].apply(lambda x: x[2])
dataset_new.df["size_y"] = dataset_new.df["bbox"].apply(lambda x: x[3])
dataset_new.df["size"] = dataset_new.df[["size_x", "size_y"]].min(axis=1)

In [ ]:
n1 = 4
n2 = 2
str_replace = {
    "FLIPPER_FL": "ALF",
    "FLIPPER_FR": "ARF",
    "FLIPPER_RL": "RLF",
    "FLIPPER_RR": "RRF",
}
for name in names:
    for identity, df_identity in tqdm(dataset_new.df.groupby("identity")):
        if identity not in new_identities or identity == "unknown":
            continue
        idx1 = (~df_identity["score"].isnull()) * (df_identity["label"] == name)
        idx2 = (df_identity["size"] >= min_size) * (df_identity["score"] >= min_score)
        df_identity = df_identity[idx1 * idx2]
        if len(df_identity) >= n1:
            df_identity = df_identity.sort_values("score", ascending=False).iloc[:n1]
        df_identity = df_identity.sort_values("size", ascending=False)
        for i in range(min(len(df_identity), n2)):
            j = df_identity.index[i]
            code = dataset_new.extract_code(j)
            for str1, str2 in str_replace.items():
                code = code.replace(str1, str2)
            dataset_new[j].save(f"{root_results}/{code}_{i}_{df_identity.loc[j, 'score']}.jpg")